<a href="https://colab.research.google.com/github/shamshamini2005-creator/NM---PROJECT-TITLE-/blob/main/SQL_generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
pip install -U transformers

In [14]:
#Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-generation", model="ibm-granite/granite-3.3-2b-instruct")
messages = [
    {"role": "user", "content": "who are you?"}
]
pipe(messages)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/787 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

...

added_tokens.json:   0%|          | 0.00/207 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': [{'role': 'user', 'content': 'who are you?'},
   {'role': 'assistant',
    'content': 'I am an assistant designed to help answer your questions and provide information. I strive to provide concise and accurate responses in English.'}]}]

In [16]:
# Load model directly

from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("ibm-granite/granite-3.3-2b-instruct")

model = AutoModelForCausalLM.from_pretrained("ibm-granite/granite-3.3-2b-instruct")

messages = [
    {"role": "user", "content": "who are you?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=48)

print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

I am an assistant designed to help answer your questions and provide information. I strive to provide concise and accurate responses in English.<|end_of_text|>


In [17]:
!pip install -q gradio transformers torch black autopep8 reportlab requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.9/88.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 20.6 MB/s eta 0:00:00


In [32]:
!pip install -q gradio transformers torch black autopep8 reportlab requests
import gradio as gr

from datetime import datetime

from pathlib import Path

import json

import sqlite3

import re

from reportlab.lib.pagesizes import letter

from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak, Table, TableStyle

from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle

from reportlab.lib.units import inch

from reportlab.lib import colors

import requests

import urllib.parse

#DATABASE SCHEMA
SCHEMA = {
    'users': {
        'id': 'INTEGER PRIMARY KEY',
        'name': 'TEXT'
    },
    'orders': {
        'id': 'INTEGER PRIMARY KEY',
        'user_id': 'INTEGER',
        'product_name': 'TEXT',
        'amount': 'REAL',
        'order_date': 'DATE',
        'status': 'TEXT'
    },
    'products': {
        'id': 'INTEGER PRIMARY KEY',
        'name': 'TEXT',
        'price': 'REAL',
        'category': 'TEXT',
        'stock': 'INTEGER'
    },
    'transactions': {
        'id': 'INTEGER PRIMARY KEY',
        'user_id': 'INTEGER',
        'amount': 'REAL',
        'date': 'DATE',
        'type': 'TEXT'
    }
}

#SECURITY & SQL GENERATION
DANGEROUS_KEYWORDS = ['DROP', 'DELETE', 'ALTER', 'TRUNCATE', 'INSERT', 'UPDATE', 'EXEC', 'EXECUTE']
DANGEROUS_CHARS = ['x', 'sp_']

def is_safe(text):
    """Check if text contains dangerous SQL keywords"""
    text_upper = text.upper()

    for keyword in DANGEROUS_KEYWORDS:
        if keyword in text_upper:
            return False

    for char in DANGEROUS_CHARS:
        if char in text:
            return False

    return True

def get_table_name(query_text):
    """Intelligently infer table name from natural language"""
    query_lower = query_text.lower()

    keyword_map = {
        'users': ['user', 'users', 'customer', 'customers', 'account', 'accounts', 'profile'],
        'orders': ['order', 'orders', 'purchase', 'purchases', 'buy', 'bought'],
        'products': ['product', 'products', 'item', 'items', 'goods'],
        'transactions': ['transaction', 'transactions', 'payment', 'payments', 'transfer']
    }
    for table, keywords in keyword_map.items():
        for keyword in keywords:
            if keyword in query_lower:
                return table
    return None # If no table is found

def get_column_names(table, query_text):
    """Intelligently infer column names from natural language based on schema"""
    query_lower = query_text.lower()
    selected_cols = []

    if table not in SCHEMA:
        return ['*'] # Default to all columns if table schema not found

    table_schema = SCHEMA[table]
    # Simple mapping, can be expanded
    column_keyword_map = {
        'id': ['id', 'identifier'],
        'name': ['name', 'title'],
        'user_id': ['user id', 'customer id'],
        'product_name': ['product name', 'item name'],
        'amount': ['amount', 'price', 'value'],
        'order_date': ['order date', 'date of order'],
        'status': ['status', 'state'],
        'price': ['price', 'cost'],
        'category': ['category', 'type'],
        'stock': ['stock', 'quantity']
    }

    # Filter column_keyword_map to only include columns relevant to the current table
    relevant_column_keywords = {col: keywords for col, keywords in column_keyword_map.items() if col in table_schema}

    for col, keywords in relevant_column_keywords.items():
        for keyword in keywords:
            if keyword in query_lower:
                selected_cols.append(col)
                break # Found a keyword for this column, move to next column

    if not selected_cols:
        return ['*'] # Default to all columns if no specific columns are mentioned
    else:
        return list(dict.fromkeys(selected_cols)) # Remove duplicates

def extract_conditions(query_text):

    """Extract WHERE conditions from natural language"""

    query_lower = query_text.lower()

    conditions = []

    #Last month
    if 'last month' in query_lower:
        conditions.append("order_date >= DATE('now', '-1 month')")

    # Last year
    if 'last year' in query_lower or 'this year' in query_lower:
        conditions.append("order_date >= DATE('now', '-1 year')")

    #Specific date patterns
    if 'last week' in query_lower:
        conditions.append("order_date >= DATE('now', '-7 days')")

    if 'today' in query_lower:
        conditions.append("order_date = DATE('now')")

    #Active/inactive
    if 'active' in query_lower:
        conditions.append("status = 'active'")
    elif 'inactive' in query_lower:
        conditions.append("status = 'inactive'")

    #Amount filtering
    amount_match = re.search(r'more than|greater than|above\s+([0-9]+)', query_lower)
    if amount_match:
        conditions.append(f"amount > {amount_match.group(1)}")

    # Less than filtering
    amount_match_lt = re.search(r'less than|below\s+([0-9]+)', query_lower)
    if amount_match_lt:
        conditions.append(f"amount < {amount_match_lt.group(1)}")

    return conditions

def generate_sql(query_text):

    """Generate SQLite SQL from natural language"""

    if not is_safe(query_text):

        return None, "X UNSAFE: Query contains dangerous SQL keywords!"

    table = get_table_name(query_text)
    if not table:
        return None, "No relevant table found in the query."

    columns = get_column_names(table, query_text)
    columns_str = ", ".join(columns) if columns else "*"

    #Build base query
    sql = f"SELECT {columns_str} FROM {table}"

    #Add conditions
    conditions = extract_conditions(query_text)
    if conditions:
        sql += " WHERE " + " AND ".join(conditions)

    #Add limit
    if 'all' not in query_text.lower():
        sql += " LIMIT 10"

    return sql, "SQL generated successfully."

query_history = []

def process_query(user_query, chat_history):
    """Process database query request"""
    if not user_query.strip():
        return chat_history

    chat_history.append((user_query, "Processing..."))

    sql_query, explanation = generate_sql(user_query)

    if sql_query is None:
        response = f"X ERROR\n\n{explanation}"
    else:
        response = f"SQL GENERATED\n\n```sql\n{sql_query}\n```\n\n{explanation}"

    query_history.append({
        'user': user_query,
        'sql': sql_query,
        'timestamp': datetime.now().isoformat()
    })
    chat_history[-1] = (user_query, response)
    return chat_history

def clear_chat():
    global query_history
    query_history = []
    return None # Gradio chatbot expects a return value for output component, None clears it

def download_pdf():
    if not query_history:
        return "X No queries to export!", None

    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    filename = f"sql_queries_{timestamp}.pdf"

    doc = SimpleDocTemplate(filename, pagesize=letter)
    styles = getSampleStyleSheet()
    story = []

    try:
        title_style = ParagraphStyle(
            'CustomTitle',
            parent=styles['Heading1'],
            fontSize=26,
            textColor=colors.HexColor('#FF6868'), # Assuming #FF6868 is a hex color
            spaceAfter=30,
            alignment=1 # CENTER
        )
        story.append(Paragraph("SQL Query Report", title_style))
        story.append(Paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", styles['Normal']))
        story.append(Spacer(1, 0.3 * inch))

        for i, query in enumerate(query_history, 1):
            story.append(Paragraph(f"<b>Query #{i}</b>", styles['Heading2']))
            story.append(Paragraph(f"User Request: {query['user']}", styles['Normal']))
            # Using a monospaced font for SQL for better readability
            story.append(Paragraph(f"Generated SQL: <font face='Courier'>{query['sql']}</font>", styles['Code']))
            story.append(Paragraph(f"Timestamp: {query['timestamp']}", styles['Small']))
            story.append(Spacer(1, 0.1 * inch))

        doc.build(story)
        return f"✓ PDF Downloaded: {filename}", filename
    except Exception as e:
        return f"X Error: {str(e)}", None

def download_txt():
    if not query_history:
        return "X No queries to export!", None

    content = f"SQL Query History Report\nGenerated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n{'='*60}\n\n"

    for i, query in enumerate(query_history, 1):
        content += f"Query #{i}\n"
        content += f"User Request: {query['user']}\n"
        content += f"Generated SQL: {query['sql']}\n"
        content += f"Timestamp: {query['timestamp']}\n"
        content += f"{'-'*60}\n\n"

    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    filename = f"sql_queries_{timestamp}.txt"

    try:
        Path(filename).write_text(content)
        return f"✓ TXT Downloaded: {filename}", filename
    except Exception as e:
        return f"X Error: {str(e)}", None

def generate_whatsapp_link():
    if not query_history:
        return "X No queries to share!"

    message = "SQL Queries Generated:\n\n"
    recent_queries = query_history[-5:]

    for i, query in enumerate(recent_queries, 1):
        message += f"*Query {i}: {query['user']}\n"
        if query['sql']:
            message += f"```{query['sql']}```\n\n"
        else:
            message += "(SQL generation failed)\n\n"

    encoded_message = urllib.parse.quote(message)
    whatsapp_link = f"https://wa.me/?text={encoded_message}"

    return f"✓ WhatsApp Link Created!\n\n{whatsapp_link}\n\nClick to open WhatsApp and share!"

def generate_facebook_link():
    if not query_history:
        return "X No queries to share!"

    message = "SQL Queries Generated:\n\n"
    recent_queries = query_history[-5:]

    for i, query in enumerate(recent_queries, 1):
        message += f"Query {i}: {query['user']}\n"
        if query['sql']:
            message += f"{query['sql']}\n\n"
        else:
            message += "(SQL generation failed)\n\n"

    encoded_message = urllib.parse.quote(message)
    facebook_link = f"https://www.facebook.com/sharer/sharer.php?quote={encoded_message}"

    return f"✓ Facebook Share Link Created!\n\n{facebook_link}\n\nClick to open Facebook and share!"

with gr.Blocks(title="SQL Query Generator") as demo:
    gr.Markdown("## Natural Language → SQLite SQL")
    gr.Markdown("Convert plain English requests into safe, production-ready SQL queries!")

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                label="Query Conversion",
                height=450,
                show_copy_button=True,
                bubble_full_width=False
            )
            with gr.Row():
                query_input = gr.Textbox(
                    label="Natural Language Request",
                    placeholder="e.g., Show me all users who signed up last month",
                    lines=2,
                    scale=4
                )
                send_btn = gr.Button("Generate SQL", scale=1, size="lg", variant="primary")

        with gr.Column(scale=1):
            gr.Markdown("### Controls")
            clear_btn = gr.Button("🗑️ Clear History", variant="stop", size="sm")
            gr.Markdown("-----------")
            gr.Markdown("### Export")
            pdf_output = gr.File(label="Download PDF", file_count='single', visible=False)
            pdf_btn = gr.Button("⬇️ Download PDF", size="sm", variant="primary")
            txt_output = gr.File(label="Download TXT", file_count='single', visible=False)
            txt_btn = gr.Button("⬇️ Download TXT", size="sm", variant="primary")
            gr.Markdown("-----------")
            gr.Markdown("### Share")
            whatsapp_btn = gr.Button("📱 Whatsapp", size="sm", variant="secondary")
            facebook_btn = gr.Button("📘 Facebook", size="sm", variant="secondary")
            share_output = gr.Textbox(label="Share Link", interactive=False, lines=5)

    # Event handlers
    send_btn.click(process_query, inputs=[query_input, chatbot], outputs=[chatbot])
    query_input.submit(process_query, inputs=[query_input, chatbot], outputs=[chatbot])
    clear_btn.click(clear_chat, outputs=[chatbot])
    pdf_btn.click(download_pdf, outputs=[share_output, pdf_output])
    txt_btn.click(download_txt, outputs=[share_output, txt_output])
    whatsapp_btn.click(generate_whatsapp_link, outputs=[share_output])
    facebook_btn.click(generate_facebook_link, outputs=[share_output])

    gr.Markdown("-----------")
    gr.Markdown("### Schema Info")
    gr.Markdown("**Tables:**\n- `users`\n- `orders`\n- `products`\n- `transactions`")
    gr.Markdown("**Supported Requests:**\n- User queries (last month, etc)")

demo.launch(share=True, show_error=True, show_api=False)

/tmp/ipykernel_350/501375961.py:331: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_350/501375961.py:331: DeprecationWarning: The 'show_copy_button' parameter will be removed in Gradio 6.0. You will need to use 'buttons=["copy"]' instead.
  chatbot = gr.Chatbot(
/tmp/ipykernel_350/501375961.py:331: DeprecationWarning: The 'bubble_full_width' parameter will be removed in Gradio 6.0. This parameter no longer has any effect.
  chatbot = gr.Chatbot(
/tmp/ipykernel_350/501375961.py:331: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  ch

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fd94527b8a20733446.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
